In [3]:
import pandas as pd

print("Iniciando pipeline de processamento - Dados ENEM 2024...")

# Seleção das features necessárias para a análise
cols = ['NU_SEQUENCIAL', 'NO_MUNICIPIO_PROVA', 'TP_DEPENDENCIA_ADM_ESC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']

# Extração (Extract)
df = pd.read_csv('/content/drive/MyDrive/Projeto ENEM/RESULTADOS_2024.csv', sep=';', encoding='latin1', usecols=cols)

# Filtragem de escopo geográfico (Assis-SP)
df_assis = df[df['NO_MUNICIPIO_PROVA'] == 'Assis'].copy()
print(f"Total de registros locais recuperados: {len(df_assis)}")

# Transformação (Transform) e Limpeza
# Remoção de registros nulos (candidatos ausentes nas provas específicas)
df_clean = df_assis.dropna(subset=['NU_NOTA_MT', 'NU_NOTA_REDACAO']).copy()

# Remoção de registros sem identificação de dependência administrativa (treineiros/egressos)
df_clean = df_clean.dropna(subset=['TP_DEPENDENCIA_ADM_ESC']).copy()

# Padronização da variável categórica de rede de ensino (2 = Pública, 3 = Privada)
df_clean['TP_ESCOLA'] = df_clean['TP_DEPENDENCIA_ADM_ESC'].apply(lambda x: 3 if x == 4 else 2)
df_clean = df_clean.rename(columns={'NU_SEQUENCIAL': 'NU_INSCRICAO'})

print(f"Amostra final após tratamento de dados (N): {len(df_clean)}\n")

# Análise Exploratória (EDA)
media_mt = df_clean.groupby('TP_ESCOLA')['NU_NOTA_MT'].mean()
media_red = df_clean.groupby('TP_ESCOLA')['NU_NOTA_REDACAO'].mean()

print("--- ESTATÍSTICAS DESCRITIVAS ---")
print(f"Competência Matemática | Rede Pública: {media_mt.get(2, 0):.1f} | Rede Privada: {media_mt.get(3, 0):.1f}")
print(f"Competência Redação    | Rede Pública: {media_red.get(2, 0):.1f} | Rede Privada: {media_red.get(3, 0):.1f}\n")

# Carga (Load) - Exportação do dataset tratado
colunas_finais = ['NU_INSCRICAO', 'NO_MUNICIPIO_PROVA', 'TP_ESCOLA', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']
df_clean[colunas_finais].to_csv('enem_assis_tratado.csv', index=False)
print("Pipeline finalizado. Arquivo CSV exportado com sucesso.")

Iniciando pipeline de processamento - Dados ENEM 2024...
Total de registros locais recuperados: 2200
Amostra final após tratamento de dados (N): 845

--- ESTATÍSTICAS DESCRITIVAS ---
Competência Matemática | Rede Pública: 517.8 | Rede Privada: 610.9
Competência Redação    | Rede Pública: 621.9 | Rede Privada: 756.1

Pipeline finalizado. Arquivo CSV exportado com sucesso.
